# Champion Pick Rates with Bootstrapping

**Question:** how much do I really play my most picked champion?

My last games are only a sample of all the games I play. If I looked at a different
set of games the pick rate would come out a little different. In this notebook I
use bootstrapping (from DSC 10) to make a 95% confidence interval for the pick rate
of my most played champion.

Run `python get_games.py` first to download the games into `games.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Load my games

In [ ]:
games = pd.read_csv("games.csv")

print("number of games:", games.shape[0])
games.head()

## 2. How many times I picked each champion

Champion is a categorical variable so this one is a bar chart.

In [ ]:
pick_counts = games.groupby("champion").count()
pick_counts = pick_counts.sort_values(by="win", ascending=False)
pick_counts = pick_counts.assign(games_played=pick_counts.get("win"))
pick_counts = pick_counts.get(["games_played"])
pick_counts

In [ ]:
top_10 = pick_counts.head(10)

plt.figure(figsize=(8, 5))
plt.barh(top_10.index, top_10.get("games_played"))
plt.gca().invert_yaxis()  # most picked champion on top
plt.xlabel("Games played")
plt.title("My 10 most picked champions")
plt.show()

## 3. Pick rate of my most played champion

In [ ]:
top_champion = pick_counts.index[0]
num_games = games.shape[0]

observed_rate = pick_counts.get("games_played").iloc[0] / num_games

print("most picked champion:", top_champion)
print("pick rate in my sample:", round(observed_rate * 100, 1), "%")

## 4. Bootstrap the pick rate

I resample my games **with replacement**, using the same number of games as the
original sample, and calculate the pick rate each time. I do this 5000 times.

In [ ]:
repetitions = 5000
boot_rates = np.array([])

for i in np.arange(repetitions):
    resample = games.sample(num_games, replace=True)
    picked = resample[resample.get("champion") == top_champion].shape[0]
    boot_rates = np.append(boot_rates, picked / num_games)

boot_rates[:10]

## 5. 95% confidence interval

The middle 95% of the bootstrapped pick rates, so I take the 2.5th and 97.5th percentiles.

In [ ]:
left = np.percentile(boot_rates, 2.5)
right = np.percentile(boot_rates, 97.5)

print("95% confidence interval:", round(left * 100, 1), "% to", round(right * 100, 1), "%")

## 6. Histogram of the bootstrapped pick rates

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(boot_rates * 100, bins=20, density=True, ec="w")
plt.axvline(left * 100, color="gold", linewidth=3, label="95% confidence interval")
plt.axvline(right * 100, color="gold", linewidth=3)
plt.axvline(observed_rate * 100, color="red", linestyle="--", label="pick rate in my sample")
plt.xlabel(top_champion + " pick rate (%)")
plt.ylabel("Density")
plt.title("Bootstrapped pick rates for " + top_champion)
plt.legend()
plt.show()

## What I found

*(fill this in after running the notebook with your own numbers)*

- My most picked champion was ___ and I played it in ___% of my games.
- I am 95% confident that my true pick rate is between ___% and ___%.
- The interval is wide/narrow because ___.

**Limits:** my recent games are not really a random sample of all my games (they
are just the newest ones), and with a small number of games the interval is wide.